Imports and Model Loading

In [3]:
import json
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

print("Loading sentence-transformers model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded successfully!")

Loading sentence-transformers model...
Model loaded successfully!


Load and Read History File

In [10]:
FILE_PATH = "../data/History.json"

with open(FILE_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

history_list = raw_data.get("Browser History", [])
records = []
# Only process first 100 entries for testing
for item in history_list[:100]: 
    title = item.get("title", "")
    url = item.get("url", "")
    if title and not title.startswith("http"):
        records.append({
            "title": title,
            "url": url,
            "timestamp": item.get("time_usec")
        })
        
# Confirm first 5 records match
for i in range(5):
    print(history_list[i])
    print(records[i])

{'favicon_url': 'https://www.google.com/favicon.ico', 'page_transition_qualifier': 'CLIENT_REDIRECT', 'title': 'Google Takeout', 'url': 'https://takeout.google.com/?pli=1', 'time_usec': 1780095359105635, 'client_id': 'fVWQ72r2tLxqGX6iXOPOfw=='}
{'title': 'Google Takeout', 'url': 'https://takeout.google.com/?pli=1', 'timestamp': 1780095359105635}
{'favicon_url': 'https://www.google.com/favicon.ico', 'page_transition_qualifier': 'CLIENT_REDIRECT', 'title': 'Google Takeout', 'url': 'https://takeout.google.com/?pli=1', 'time_usec': 1780095314642711, 'client_id': 'fVWQ72r2tLxqGX6iXOPOfw=='}
{'title': 'Google Takeout', 'url': 'https://takeout.google.com/?pli=1', 'timestamp': 1780095314642711}
{'favicon_url': 'https://www.google.com/favicon.ico', 'page_transition_qualifier': 'CLIENT_REDIRECT', 'title': 'Google Takeout', 'url': 'https://takeout.google.com/?pli=1', 'time_usec': 1780095312972114, 'client_id': 'fVWQ72r2tLxqGX6iXOPOfw=='}
{'title': 'Google Takeout', 'url': 'https://takeout.google.

Create DataFrame and Remove Duplicate Data

In [9]:
df = pd.DataFrame(records)
df = df.drop_duplicates(subset=['title']).reset_index(drop=True)

print(f"Total rows processed: {len(df)}")
# Inspect the top 5 entries to make sure it looks right
df.head()

Total rows processed: 42


,title,url,timestamp
0,Google Takeout,https://takeout.google.com/?pli=1,1780095359105635
1,General,https://github.com/Blairqiao/Spotify_Playlist_...,1780095217347734
2,Blairqiao/Spotify_Playlist_Visualizer,https://github.com/Blairqiao/Spotify_Playlist_...,1780095215934324
3,google takeout - Google Search,https://www.google.com/search?q=google+takeout...,1780094752425801
4,simon-th/spotify-data-project: Analyzing Spoti...,https://github.com/simon-th/spotify-data-project,1780094120575384


Create High-dimensional Embeddings for NLP Model

In [11]:
print("Encoding titles into high-dimensional vectors...")
titles_list = df['title'].tolist()
embeddings = model.encode(titles_list, show_progress_bar=True)

print("\nEmbedding Generation Complete!")
print(f"Shape of vector matrix: {embeddings.shape}") 
# Expected shape: (number_of_unique_titles, 384)

Encoding titles into high-dimensional vectors...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


Embedding Generation Complete!
Shape of vector matrix: (42, 384)


Using Cosine Similarity to Check Model Understanding

In [21]:
from sklearn.metrics.pairwise import cosine_similarity

# Pick the very first title in your history
sample_index = 0
sample_vector = embeddings[sample_index].reshape(1, -1)

# Calculate similarity scores against all other vectors
scores = cosine_similarity(sample_vector, embeddings)[0]

# Add scores to a temporary dataframe to view results
test_df = df.copy()
test_df['similarity'] = scores
test_df = test_df.sort_values(by='similarity', ascending=False)

print(f"Target Title: '{df.iloc[sample_index]['title']}'\n")
print("Top 3 most mathematically similar pages found in your history:")
test_df[['title', 'similarity']].iloc[:4]

Target Title: 'Google Takeout'

Top 3 most mathematically similar pages found in your history:


,title,similarity
0,Google Takeout,1.000000
3,google takeout - Google Search,0.899606
18,reccobeats - Google Search,0.264273
40,spotify audio features deprecated alternative ...,0.261095
